<a href="https://colab.research.google.com/github/Reginajose/Observatorio-Clima-Grande-desafio/blob/main/C%C3%B3pia_de_Analise_dos_Dados_do_Atlas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise Exploratória e Diagnóstico de Qualidade (Sprint 1)
**Objetivo:** Este caderno (notebook) não altera nem corrige os dados. O objetivo aqui é apenas "tirar uma radiografia" da base de dados do Atlas Digital de Desastres.

Vamos carregar o arquivo original, aplicar o filtro do nosso escopo (Região Sul, 2021 a 2025, Desastres Hidrológicos) e gerar um relatório apontando o que está consistente e o que tem problemas (dados nulos, em branco ou mal formatados), para documentar no nosso Data Card.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Importamos a biblioteca 'pandas', que é a ferramenta padrão do Python para ler e analisar tabelas
import pandas as pd

# 1. Carregamento da base de dados
# Indicamos o nome exato do arquivo. O 'sep=";"' avisa ao computador que as colunas no Brasil são separadas por ponto e vírgula
caminho_arquivo = '/content/drive/MyDrive/atlas_geral.csv'

print("Lendo o arquivo original...")
df_bruto = pd.read_csv(caminho_arquivo, sep=';', encoding='utf-8')

print(f"Base carregada com sucesso! O arquivo inteiro tem {df_bruto.shape[0]} linhas e {df_bruto.shape[1]} colunas.")

Lendo o arquivo original...
Base carregada com sucesso! O arquivo inteiro tem 76190 linhas e 70 colunas.


/tmp/ipykernel_3491/1326060330.py:9: DtypeWarning:

Columns (11,43,44,45,46,47,48,49,62) have mixed types. Specify dtype option on import or set low_memory=False.



## 1. Aplicando o Recorte do Projeto
A base contém o Brasil inteiro desde 1991. Para avaliar a qualidade do dado que *nós* vamos usar, primeiro precisamos isolar apenas o nosso alvo: estados do Sul (RS, SC, PR), anos de 2021 a 2025, e o grupo Hidrológico (que engloba Inundações, Enxurradas, Alagamentos, Chuvas Intensas e Movimento de Massa).

In [ ]:
df_bruto['Data_Evento_dt'] = pd.to_datetime(df_bruto['Data_Evento'], format='%d/%m/%Y', errors='coerce')

filtro_estado = df_bruto['Sigla_UF'].isin(['PR', 'RS', 'SC'])
filtro_ano = (df_bruto['Data_Evento_dt'].dt.year >= 2021) & (df_bruto['Data_Evento_dt'].dt.year <= 2025)
filtro_grupo = df_bruto['grupo_de_desastre'] == 'Hidrológico'

df_recorte = df_bruto[filtro_estado & filtro_ano & filtro_grupo].copy()

print(f"Pronto! Nosso recorte tem exatamente {len(df_recorte)} registros de desastres.")

Pronto! Nosso recorte tem exatamente 3756 registros de desastres.


## 2. Estado Atual da Base (Distribuição Geográfica e Tipologias)
Agora vamos pedir para o computador contar quantas linhas existem para cada estado, quais os tipos exatos de desastres que sobraram no recorte e qual o status de reconhecimento deles no sistema S2iD.

In [ ]:
# O comando 'value_counts()' agrupa os itens iguais e conta quantos existem de cada
print("--- DISTRIBUIÇÃO POR ESTADO ---")
print(df_recorte['Sigla_UF'].value_counts())

print("\n--- QUAIS TIPOS DE DESASTRES EXISTEM NESTE RECORTE? ---")
print(df_recorte['descricao_tipologia'].value_counts())

print("\n--- STATUS LEGAL DOS DESASTRES ---")
# Mostra se o desastre foi apenas registrado pela prefeitura ou se já foi reconhecido oficialmente pelo Governo
print(df_recorte['Status'].value_counts())

--- DISTRIBUIÇÃO POR ESTADO ---
Sigla_UF
SC    1989
RS    1517
PR     250
Name: count, dtype: int64

--- QUAIS TIPOS DE DESASTRES EXISTEM NESTE RECORTE? ---
descricao_tipologia
Chuvas Intensas       2630
Enxurradas             609
Alagamentos            220
Inundações             192
Movimento de Massa     105
Name: count, dtype: int64

--- STATUS LEGAL DOS DESASTRES ---
Status
Registro       1984
Reconhecido    1772
Name: count, dtype: int64


In [ ]:
import plotly.graph_objects as go

def grafico_barra_historia(categorias, valores, titulo, subtitulo, cor_destaque='#C0392B', destaque=None, height=None, formatador=None):
    """
    Gráfico de barras horizontais no estilo Storytelling com Dados:
    ordenado por valor, cinza neutro + 1 cor de destaque, rótulo direto, sem grid.
    'destaque' = índice da barra que sustenta o título (None = destaca a maior automaticamente).
    'formatador' = função opcional para formatar os rótulos de valores.
    """
    ordem = sorted(range(len(valores)), key=lambda i: valores[i])  # menor pro maior (barra maior fica no topo)
    categorias_ord = [categorias[i] for i in ordem]
    valores_ord = [valores[i] for i in ordem]
    idx_destaque = destaque if destaque is not None else valores.index(max(valores))
    cores = [cor_destaque if i == idx_destaque else '#B8BCC2' for i in ordem]

    if formatador:
        rotulos = [formatador(v) for v in valores_ord]
    else:
        total = sum(valores)
        rotulos = [f"{v} ({v/total*100:.0f}%)" for v in valores_ord]

    fig = go.Figure(go.Bar(
        x=valores_ord, y=categorias_ord, orientation='h',
        marker_color=cores, text=rotulos, textposition='outside',
        textfont=dict(size=14, color='#2E2E2E'),
    ))
    fig.update_layout(
        title=dict(
            text=f"<b>{titulo}</b><br><span style='font-size:13px;color:#6B6B6B'>{subtitulo}</span>",
            x=0.02, xanchor='left'
        ),
        xaxis=dict(showgrid=False, showticklabels=False, zeroline=False, title=None, range=[0, max(valores)*1.3]),
        yaxis=dict(showgrid=False, title=None, tickfont=dict(size=13)),
        plot_bgcolor='white', paper_bgcolor='white', showlegend=False,
        margin=dict(l=10, r=60, t=85, b=10),
        height=height or (100 + 45*len(categorias)),
    )
    fig.show()

# --- Gráfico 1: Distribuição por estado ---
uf = df_recorte['Sigla_UF'].value_counts()
grafico_barra_historia(
    categorias=uf.index.tolist(), valores=uf.values.tolist(),
    titulo="Santa Catarina sozinho tem mais registros que PR e RS juntos",
    subtitulo="Distribuição de desastres hidrológicos por estado — RS/SC/PR, 2021–2025"
)

# --- Gráfico 2: Tipos de desastre ---
tipo = df_recorte['descricao_tipologia'].value_counts()
grafico_barra_historia(
    categorias=tipo.index.tolist(), valores=tipo.values.tolist(),
    titulo="7 em cada 10 registros são de Chuvas Intensas",
    subtitulo="Distribuição por tipologia dentro do grupo Hidrológico — RS/SC/PR, 2021–2025"
)

# --- Gráfico 3: Status ---
status = df_recorte['Status'].value_counts()
grafico_barra_historia(
    categorias=status.index.tolist(), valores=status.values.tolist(),
    titulo="Mais da metade dos registros ainda não foi homologada",
    subtitulo="Status legal dos desastres — RS/SC/PR, 2021–2025",
    cor_destaque='#C0392B', destaque=list(status.index).index('Registro')
)

## 3. Diagnóstico de Qualidade: Identificando Valores Nulos (Vazios)
Um dos maiores problemas em análise de dados é a falta de preenchimento. Se colunas vitais como o Código IBGE, Danos Materiais ou População Afetada estiverem em branco, nosso cruzamento com o MapBiomas vai falhar. Aqui vamos apenas contar e expor quantos buracos existem na base.

In [ ]:
# Selecionamos as colunas relevantes para o nosso projeto
# Status foi adicionada por recomendação explícita do nosso próprio Data Card (Sprint 1, seção 3):
# a omissão de valor financeiro está associada ao Status do registro, e precisa aparecer como contexto no ranking.
colunas_vitais = [
    'Cod_IBGE_Mun',
    'Nome_Municipio',
    'Sigla_UF',
    'Data_Evento',
    'Status',                      # NOVO — contexto de homologação, associado ao viés de "falsos zeros"
    'descricao_tipologia',         # NOVO — opcional, útil pro SHOULD (composição do Top 50 por tipo de evento)
    'Protocolo_S2iD',              # NOVO — opcional, rastreabilidade/auditoria de qualquer valor estranho
    'DM_total_danos_materiais',
    'PEPL_total_publico',
    'DH_MORTOS',
    'DH_DESALOJADOS',
    'DH_total_danos_humanos_diretos'   # NOVO — total oficial, serve de conferência das categorias acima
]

nulos_encontrados = df_recorte[colunas_vitais].isnull().sum()

print("--- RELATÓRIO DE BURACOS (VALORES NULOS) ---")
print(nulos_encontrados)

datas_invalidas = df_recorte['Data_Evento_dt'].isnull().sum()
print(f"\nExistem {datas_invalidas} registros com datas impossíveis de ler pelo computador.")

--- RELATÓRIO DE BURACOS (VALORES NULOS) ---
Cod_IBGE_Mun                      0
Nome_Municipio                    0
Sigla_UF                          0
Data_Evento                       0
Status                            0
descricao_tipologia               0
Protocolo_S2iD                    0
DM_total_danos_materiais          0
PEPL_total_publico                0
DH_MORTOS                         0
DH_DESALOJADOS                    0
DH_total_danos_humanos_diretos    0
dtype: int64

Existem 0 registros com datas impossíveis de ler pelo computador.


## 4. Diagnóstico de Qualidade: Identificando Dados Sujos (Inconsistências)
Às vezes o dado não está nulo, mas está digitado errado. Um erro clássico de prefeituras é colocar um espaço em branco no final do nome da cidade (ex: digitar `"Curitiba "` em vez de `"Curitiba"`). O olho humano não vê diferença, mas o computador considera que são duas cidades diferentes. Vamos rastrear isso.

In [ ]:
# O código abaixo cria uma versão invisível da coluna de municípios removendo espaços inúteis no início e no fim (.str.strip())
# Depois, ele compara a versão limpa com a versão original. Se forem diferentes, é porque o original estava sujo.
municipios_originais = df_recorte['Nome_Municipio'].astype(str)
municipios_sem_espaco = municipios_originais.str.strip()

# Contamos quantas vezes o original é diferente do limpo
quantidade_sujos = (municipios_originais != municipios_sem_espaco).sum()

print("--- RELATÓRIO DE DADOS SUJOS (ESPAÇOS INVISÍVEIS) ---")
print(f"Foram encontrados {quantidade_sujos} registros onde o nome do município tem espaços em branco extras sobrando.")

--- RELATÓRIO DE DADOS SUJOS (ESPAÇOS INVISÍVEIS) ---
Foram encontrados 0 registros onde o nome do município tem espaços em branco extras sobrando.


## 5. Resumo das Colunas Financeiras e de Impacto
Por fim, vamos olhar para o panorama dos valores declarados (em Reais e em Vidas). O comando `describe()` nos mostra qual foi a média de prejuízo por desastre, qual foi o valor máximo declarado, se existem valores negativos (o que seria um erro grave) e como os números se distribuem.

In [ ]:
colunas_impacto = [
    'DM_total_danos_materiais',
    'PEPL_total_publico',
    'DH_MORTOS',
    'DH_DESALOJADOS',
    'DH_DESABRIGADOS',
    'DH_ENFERMOS',
    'DH_DESAPARECIDOS',
    'DH_total_danos_humanos_diretos'   # NOVO — total oficial, serve de conferência das categorias acima
]

df_impacto_numerico = df_recorte[colunas_impacto].apply(pd.to_numeric, errors='coerce')

pd.options.display.float_format = '{:,.2f}'.format

print("--- ESTATÍSTICAS DOS DANOS E PREJUÍZOS ---")
print(df_impacto_numerico.describe())

--- ESTATÍSTICAS DOS DANOS E PREJUÍZOS ---
       DM_total_danos_materiais  PEPL_total_publico  DH_MORTOS  \
count                  3,756.00            3,756.00   3,756.00   
mean               3,673,486.23          742,814.48       0.08   
std               81,223,851.76        4,115,223.16       0.78   
min                        0.00                0.00       0.00   
25%                        0.00                0.00       0.00   
50%                   76,771.43                0.00       0.00   
75%                  874,466.92          213,490.62       0.00   
max            4,823,240,240.88      114,805,391.00      31.00   

       DH_DESALOJADOS  DH_DESABRIGADOS  DH_ENFERMOS  DH_DESAPARECIDOS  \
count        3,756.00         3,756.00     3,756.00          3,756.00   
mean           284.23            39.16         1.42              0.02   
std          3,956.53           511.98        36.96              0.57   
min              0.00             0.00         0.00              0.00 

In [ ]:
import plotly.graph_objects as go

# 1. Métrica e agregação (já devem existir de células anteriores, repito aqui pra ficar independente)
df_recorte['Custo_Dano_Evitado'] = df_recorte['DM_total_danos_materiais'] + df_recorte['PEPL_total_publico']

ranking = (df_recorte.groupby(['Cod_IBGE_Mun', 'Nome_Municipio'], as_index=False)['Custo_Dano_Evitado']
           .sum()
           .sort_values('Custo_Dano_Evitado', ascending=False))

top10 = ranking.head(10).sort_values('Custo_Dano_Evitado', ascending=True)  # ascending: no gráfico horizontal, o maior fica no topo

# 2. Calculamos a mensagem central do gráfico (o "so-what" que vira o título)
primeiro = ranking.iloc[0]
soma_proximos_9 = ranking.iloc[1:10]['Custo_Dano_Evitado'].sum()
razao = primeiro['Custo_Dano_Evitado'] / soma_proximos_9

# Adicionando o cálculo da variável 'participacao'
participacao = (primeiro['Custo_Dano_Evitado'] / ranking['Custo_Dano_Evitado'].sum()) * 100

# 3. Cor: cinza neutro pra todo mundo, destaque só pro município que sustenta a mensagem do título
cinza = '#B8BCC2'
destaque = '#C0392B'
cores = [destaque if nome == primeiro['Nome_Municipio'] else cinza for nome in top10['Nome_Municipio']]

# 4. Rótulo direto na barra (em R$ milhões, sem casas decimais desnecessárias)
rotulos = [f"R$ {v/1e6:,.0f} mi" for v in top10['Custo_Dano_Evitado']]

fig = go.Figure(go.Bar(
    x=top10['Custo_Dano_Evitado'],
    y=top10['Nome_Municipio'],
    orientation='h',
    marker_color=cores,
    text=rotulos,
    textposition='outside',
    textfont=dict(size=13, color='#2E2E2E'),
))

fig.update_layout(
    title=dict(
        text=f"<b>{primeiro['Nome_Municipio']} concentra sozinho {participacao:.0f}% do custo do Top 50</b><br>"
             f"<span style='font-size:13px;color:#6B6B6B'>{razao:.1f}x mais que a soma das 9 cidades seguintes — desastres hidrológicos, RS/SC/PR, 2021–2025</span>",
        x=0.02, xanchor='left'
    ),
    xaxis=dict(showgrid=False, showticklabels=False, zeroline=False, title=None),
    yaxis=dict(showgrid=False, title=None, tickfont=dict(size=13)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=False,
    margin=dict(l=10, r=80, t=90, b=10),
    height=430,
)

fig.show()

In [ ]:
import plotly.graph_objects as go

# Definir a função de formatação de moeda fora da estrutura da tabela
def formata_reais(v):
    if v >= 1e9:
        return f"R$ {v/1e9:,.2f} bi"
    elif v >= 1e6:
        return f"R$ {v/1e6:,.1f} mi"
    else:
        return f"R$ {v:,.0f}"

sl = df_bruto[(df_bruto['Nome_Municipio']=='São Leopoldo') & (df_bruto['Sigla_UF']=='RS') &
              (df_bruto['Data_Evento_dt'].dt.year>=2021) & (df_bruto['Data_Evento_dt'].dt.year<=2025) &
              (df_bruto['grupo_de_desastre']=='Hidrológico')].sort_values('Data_Evento_dt')

# Cor da linha: destaca o evento catastrófico, cinza pro resto
cor_linha = ['#C0392B' if v > 1e9 else '#EDEDED' for v in sl['DM_total_danos_materiais']]
cor_texto = ['white' if v > 1e9 else '#2E2E2E' for v in sl['DM_total_danos_materiais']]

fig = go.Figure(data=[go.Table(
    columnwidth=[80, 110, 90, 110],
    header=dict(
        values=['<b>Data</b>', '<b>Tipo de desastre</b>', '<b>Status</b>', '<b>Dano material</b>'],
        fill_color='#2E2E2E', font=dict(color='white', size=13), align='left', height=32
    ),
    cells=dict(
        values=[
            sl['Data_Evento'],
            sl['descricao_tipologia'],
            sl['Status'],
            [formata_reais(v) for v in sl['DM_total_danos_materiais']] # Usando a função aqui
        ],
        fill_color=[cor_linha]*4,
        font=dict(color=[cor_texto]*4, size=13),
        align='left', height=30
    )
)])

fig.update_layout(
    title=dict(
        text="<b>São Leopoldo: 1 evento catastrófico isolado, não uma recorrência</b><br>"
             "<span style='font-size:13px;color:#6B6B6B'>Todos os 4 registros de 2021–2025 já estão homologados (Reconhecido) — o valor alto não é ruído de subnotificação</span>",
        x=0.02, xanchor='left'
    ),
    margin=dict(l=10, r=10, t=90, b=10),
    height=280,
)

fig.show()

São Leopoldo lidera o ranking por causa de um único evento, a enchente de 27/04/2024, parte da catástrofe histórica do Rio Grande do Sul, quando o Rio dos Sinos atingiu 8,07m (recorde da cidade) e cerca de 180 mil pessoas foram afetadas. Os outros 3 registros da cidade no recorte são pequenos ou nulos. Importante: todos os 4 eventos já estão com Status "Reconhecido", ou seja, o valor alto não é fruto de subnotificação ou processo incompleto é um número validado. Isso responde à pergunta SHOULD do Sprint 1 sobre recorrência: São Leopoldo é caso de evento isolado catastrófico, não sofrimento contínuo ano a ano

## 6. Validação da Chave de Cruzamento (Código IBGE)
Para garantir que o nosso futuro cruzamento com a base do MapBiomas (Nível 2) não vai quebrar, precisamos garantir que todos os municípios nesta base possuem o Código IBGE no padrão de 7 dígitos.

In [ ]:
# Converte a coluna de código para texto e mede o tamanho (quantidade de caracteres) de cada linha
tamanho_ibge = df_recorte['Cod_IBGE_Mun'].astype(str).str.len()

print("--- TAMANHO DOS CÓDIGOS IBGE ---")
print(tamanho_ibge.value_counts())
# Se o resultado for apenas "7", significa que 100% da base está no formato correto para o Join!

--- TAMANHO DOS CÓDIGOS IBGE ---
Cod_IBGE_Mun
7    3756
Name: count, dtype: int64


## 7. Diagnóstico de Negócio: Os Falsos "Zero Reais"
Nossa pergunta principal do projeto (MUST) soma os danos materiais e públicos. Porém, como o S2iD depende da autodeclaração das prefeituras, muitos municípios registram o desastre mas não preenchem a estimativa em dinheiro. Vamos descobrir quantos desastres estão "zerados" na nossa base.

In [ ]:
# Filtra as linhas onde TANTO o dano material QUANTO o prejuízo público são exatamente zero
desastres_zerados = df_impacto_numerico[
    (df_impacto_numerico['DM_total_danos_materiais'] == 0) &
    (df_impacto_numerico['PEPL_total_publico'] == 0)
]

percentual_zerados = (len(desastres_zerados) / len(df_recorte)) * 100

print("--- REGISTROS SEM VALOR FINANCEIRO DECLARADO ---")
print(f"Total de desastres com R$ 0,00 declarados: {len(desastres_zerados)} ocorrências.")
print(f"Isso representa {percentual_zerados:.1f}% do nosso recorte.")
print("Nota para o Data Card: Municípios com valores zerados podem cair no final do nosso ranking, não por falta de danos, mas por falta de preenchimento institucional.")

--- REGISTROS SEM VALOR FINANCEIRO DECLARADO ---
Total de desastres com R$ 0,00 declarados: 1407 ocorrências.
Isso representa 37.5% do nosso recorte.
Nota para o Data Card: Municípios com valores zerados podem cair no final do nosso ranking, não por falta de danos, mas por falta de preenchimento institucional.


## 8. Teste de Sanidade (Top Outlier)
Encontramos um valor máximo superior a 4 Bilhões de Reais em um único registro. Vamos identificar que município e evento foi esse para garantir que o número faz sentido no mundo real e não é um erro de digitação do sistema.

In [ ]:
# O comando nlargest(1) pega a linha com o maior valor na coluna especificada
maior_desastre = df_recorte.nlargest(1, 'DM_total_danos_materiais')

print("--- O MAIOR DESASTRE DA BASE (EM DANOS MATERIAIS) ---")
print(maior_desastre[['Nome_Municipio', 'Sigla_UF', 'Data_Evento', 'descricao_tipologia', 'DM_total_danos_materiais']])

--- O MAIOR DESASTRE DA BASE (EM DANOS MATERIAIS) ---
      Nome_Municipio Sigla_UF Data_Evento descricao_tipologia  \
68750   São Leopoldo       RS  27/04/2024     Chuvas Intensas   

       DM_total_danos_materiais  
68750          4,823,240,240.88  


## 9. Exportação do Dataset Filtrado
Criação da métrica principal de custo (Danos Materiais + Prejuízos Públicos) e exportação do arquivo CSV contendo apenas o recorte de desastres hidrológicos da Região Sul (2021-2025). Este é o arquivo que será utilizado no painel.

In [ ]:
# 9. Exportação do Dataset Filtrado
# Garantir que as colunas financeiras sejam lidas como números e preencher vazios com 0
df_recorte['DM_total_danos_materiais'] = pd.to_numeric(df_recorte['DM_total_danos_materiais'], errors='coerce').fillna(0)
df_recorte['PEPL_total_publico'] = pd.to_numeric(df_recorte['PEPL_total_publico'], errors='coerce').fillna(0)

# Criar a métrica do MUST
df_recorte['Custo_Dano_Evitado'] = df_recorte['DM_total_danos_materiais'] + df_recorte['PEPL_total_publico']

# Corrigindo sujeira de espaços em branco nos nomes dos municípios ANTES de exportar
df_recorte['Nome_Municipio_Limpo'] = df_recorte['Nome_Municipio'].astype(str).str.strip().str.title()

# Filtrar apenas as colunas que a squad realmente vai usar no painel (COM A VÍRGULA CORRIGIDA)
colunas_exportacao = [
    'Protocolo_S2iD', 'Cod_IBGE_Mun', 'Nome_Municipio_Limpo', 'Sigla_UF',
    'Data_Evento', 'Cod_Cobrade', 'descricao_tipologia', 'Status',
    'DM_total_danos_materiais', 'PEPL_total_publico', 'Custo_Dano_Evitado',
    'DH_MORTOS', 'DH_DESALOJADOS', 'DH_DESABRIGADOS', 'DH_total_danos_humanos_diretos'
]

df_filtrado = df_recorte[colunas_exportacao].copy()

# Gerar o arquivo CSV
nome_arquivo = 'dataset_sul_hidrologico_2021_2025_filtrado.csv'
df_filtrado.to_csv(nome_arquivo, index=False, encoding='utf-8')

print(f"Arquivo '{nome_arquivo}' gerado com sucesso!")
print(f"Tamanho: {df_filtrado.shape[0]} linhas e {df_filtrado.shape[1]} colunas.")

Arquivo 'dataset_sul_hidrologico_2021_2025_filtrado.csv' gerado com sucesso!
Tamanho: 3756 linhas e 15 colunas.


In [ ]:
# 1) Primeiro, carrega o dataset final
df = pd.read_csv('dataset_sul_hidrologico_2021_2025_filtrado.csv')

# 2) Aqui entra a correção das preposições
import re
preposicoes = ['Do', 'Da', 'De', 'Dos', 'Das', 'E']

def corrige_preposicoes(nome):
    palavras = nome.split()
    corrigido = [p.lower() if p in preposicoes and i != 0 else p for i, p in enumerate(palavras)]
    return ' '.join(corrigido)

df['Nome_Municipio_Limpo'] = df['Nome_Municipio_Limpo'].apply(corrige_preposicoes)

restantes = df[df['Nome_Municipio_Limpo'].str.contains(r'\b(Do|Da|De|Dos|Das)\b')]['Nome_Municipio_Limpo'].nunique()
print(f"Nomes ainda com preposição maiúscula: {restantes}")

# 3) Salva o CSV corrigido, sobrescrevendo o antigo (ou com outro nome, se preferir manter as duas versões)
df.to_csv('dataset_sul_hidrologico_2021_2025_filtrado.csv', index=False)

Nomes ainda com preposição maiúscula: 0


/tmp/ipykernel_3491/1498550943.py:15: UserWarning:

This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.



## 10. Diagnóstico de Negócio: Status vs. Valor Financeiro Zerado
Além de saber que 37,5% dos registros estão com valor zerado, precisamos entender **quem** são esses registros. Será que eles se concentram nos desastres que ainda não foram formalmente reconhecidos pelo governo (Status = "Registro"), ou o problema é espalhado igualmente por toda a base? Essa checagem mostra se o "buraco" financeiro está ligado ao andamento do processo burocrático.

In [ ]:
# Reaproveitamos a mesma lógica de "zerado" usada antes:
# um registro é considerado zerado quando NENHUM dos dois valores (danos materiais e prejuízo público) foi preenchido
recorte_numerico = df_recorte[['DM_total_danos_materiais', 'PEPL_total_publico']].apply(pd.to_numeric, errors='coerce')
df_recorte['zerado'] = (recorte_numerico['DM_total_danos_materiais'] == 0) & (recorte_numerico['PEPL_total_publico'] == 0)

# O comando 'crosstab' cruza duas colunas categóricas e conta quantas vezes cada combinação aparece.
# 'normalize="index"' transforma a contagem em porcentagem DENTRO de cada Status
# (ou seja, responde: "de tudo que é Registro, que fatia está zerada?", separado de "de tudo que é Reconhecido, que fatia está zerada?")
tabela_cruzada = pd.crosstab(df_recorte['Status'], df_recorte['zerado'], normalize='index') * 100

print("--- PERCENTUAL DE REGISTROS ZERADOS, POR STATUS ---")
print(tabela_cruzada.round(1))

# Também é útil ver o número absoluto por trás dessa porcentagem, não só o percentual
print("\n--- QUANTIDADE DE REGISTROS POR STATUS (para referência) ---")
print(df_recorte['Status'].value_counts())

--- PERCENTUAL DE REGISTROS ZERADOS, POR STATUS ---
zerado       False  True 
Status                   
Reconhecido  80.30  19.70
Registro     46.70  53.30

--- QUANTIDADE DE REGISTROS POR STATUS (para referência) ---
Status
Registro       1984
Reconhecido    1772
Name: count, dtype: int64


In [ ]:
import plotly.graph_objects as go

# Re-calculando tabela_cruzada para garantir que esteja definida nesta célula
recorte_numerico = df_recorte[['DM_total_danos_materiais', 'PEPL_total_publico']].apply(pd.to_numeric, errors='coerce')
df_recorte['zerado'] = (recorte_numerico['DM_total_danos_materiais'] == 0) & (recorte_numerico['PEPL_total_publico'] == 0)
tabela_cruzada = pd.crosstab(df_recorte['Status'], df_recorte['zerado'], normalize='index') * 100

pct_registro = tabela_cruzada.loc['Registro', True]
pct_reconhecido = tabela_cruzada.loc['Reconhecido', True]
razao = pct_registro / pct_reconhecido

categorias = ['Registro<br><span style="font-size:11px;color:#8A8A8A">(não homologado)</span>',
              'Reconhecido<br><span style="font-size:11px;color:#8A8A8A">(homologado)</span>']
valores = [pct_registro, pct_reconhecido]

# Cor com propósito: destaca o problema (Registro), cinza neutro pro "controle"
cores = ['#C0392B', '#B8BCC2']
rotulos = [f"{v:.1f}%" for v in valores]

fig = go.Figure(go.Bar(
    x=valores,
    y=categorias,
    orientation='h',
    marker_color=cores,
    text=rotulos,
    textposition='outside',
    textfont=dict(size=18, color='#2E2E2E'),
    width=0.55,
))

fig.update_layout(
    title=dict(
        text=f"<b>Registro não-homologado tem {razao:.1f}x mais chance de aparecer com valor zerado</b><br>"
             f"<span style='font-size:13px;color:#6B6B6B'>% de registros com dano material E prejuízo público declarados como R$ 0 — desastres hidrológicos, RS/SC/PR, 2021–2025</span>",
        x=0.02, xanchor='left'
    ),
    xaxis=dict(showgrid=False, showticklabels=False, zeroline=False, title=None, range=[0, max(valores)*1.25]),
    yaxis=dict(showgrid=False, title=None, tickfont=dict(size=14)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=False,
    margin=dict(l=10, r=60, t=95, b=10),
    height=300,
)

fig.show()

No recorte da Região Sul, **37,5% das ocorrências (1.407 registros)** possuem valor financeiro igual a R$ 0,00. O diagnóstico revelou que a omissão de valores é um **viés burocrático de processo incompleto**: a taxa de registros zerados é **2,7 vezes maior em eventos pendentes de homologação** (`Status = Registro`, com 53,3%) do que nos já formalizados (`Status = Reconhecido`, com 19,7%). Para evitar que municípios com pendências administrativas apareçam **artificialmente baratos no ranking do MUST**, mantiveram-se 100% dos registros na base, adotando a coluna `Status` como variável de contexto.

In [ ]:
import pandas as pd
import numpy as np

# 1. Carregar o dataset
# (Ajuste o nome do arquivo se necessário)
df = pd.read_csv('dataset_sul_hidrologico_2021_2025_filtrado.csv')

# ==============================================================================
# TRATAMENTO 1: Ajuste e Padronização da Chave Cadastral (Cod_IBGE_Mun)
# ==============================================================================
# Trata o código do IBGE para garantir que seja string e sempre tenha 7 dígitos (com zeros à esquerda)
if 'Cod_IBGE_Mun' in df.columns:
    df['Cod_IBGE_Mun'] = (
        df['Cod_IBGE_Mun']
        .fillna(0)                          # Trata possíveis nulos no código
        .astype(int)                        # Converte para inteiro para remover decimais (ex: 4318705.0 -> 4318705)
        .astype(str)                        # Converte para texto
        .str.zfill(7)                       # Garante exatamente 7 dígitos (ex: 431870 -> 0431870)
    )

# ==============================================================================
# TRATAMENTO 2: Tratamento de NaN e Cálculo do Custo Total Evitado
# ==============================================================================
# Garante que as colunas financeiras sejam numéricas (converte textos inválidos em NaN)
df['DM_total_danos_materiais'] = pd.to_numeric(df['DM_total_danos_materiais'], errors='coerce')
df['PEPL_total_publico'] = pd.to_numeric(df['PEPL_total_publico'], errors='coerce')

# Substitui NaN por 0 e faz a soma perfeita sem gerar valores 'em branco'
df['DM_limpo'] = df['DM_total_danos_materiais'].fillna(0)
df['PEPL_limpo'] = df['PEPL_total_publico'].fillna(0)

# Métrica financeira final do MUST
df['Custo_Dano_Evitado'] = df['DM_limpo'] + df['PEPL_limpo']


# ==============================================================================
# AGRUPAMENTO: Criando o Ranking Top 50 Sem Cidades 'Em Branco'
# ==============================================================================
# Identifica as colunas de nome do município e UF no seu dataset
# A coluna de nome de município no dataset filtrado é 'Nome_Municipio_Limpo'
col_mun = 'Nome_Municipio_Limpo'
col_uf = 'Sigla_UF' # A coluna de UF no dataset filtrado é 'Sigla_UF'

# Garante que não haja nomes de municípios nulos
df[col_mun] = df[col_mun].fillna('Município Não Identificado')

# Agrupa por Município e UF somando os custos e contando os registros
ranking_top50 = (
    df.groupby([col_mun, col_uf, 'Cod_IBGE_Mun'], as_index=False)
    .agg(
        Custo_Total_Dano=('Custo_Dano_Evitado', 'sum'),
        Danos_Materiais=('DM_limpo', 'sum'),
        Prejuizos_Publicos=('PEPL_limpo', 'sum'),
        Total_Eventos=('Custo_Dano_Evitado', 'count')
    )
    .sort_values(by='Custo_Total_Dano', ascending=False)
    .head(50)
)

# Visualizar o resultado no Colab
display(ranking_top50.head(10))

,Nome_Municipio_Limpo,Sigla_UF,Cod_IBGE_Mun,Custo_Total_Dano,Danos_Materiais,Prejuizos_Publicos,Total_Eventos
787,São Leopoldo,RS,4318705,"5,065,996,749.38","4,987,429,994.25","78,566,755.13",4
330,Guaíba,RS,4309308,"910,494,646.56","909,469,165.43","1,025,481.13",3
3,Agrolândia,SC,4200200,"654,105,482.41","646,841,976.32","7,263,506.09",10
348,Igrejinha,RS,4310108,"318,011,685.95","297,322,719.71","20,688,966.24",7
710,Santa Vitória do Palmar,RS,4317301,"246,486,627.48","126,883,923.85","119,602,703.63",5
380,Itajaí,SC,4208203,"230,684,749.00","229,349,443.16","1,335,305.84",10
715,Santo Amaro da Imperatriz,SC,4215703,"225,882,414.51","100,730,358.89","125,152,055.62",16
225,Cruzeiro do Sul,RS,4306205,"197,796,256.85","188,983,340.48","8,812,916.37",4
423,Lajeado,RS,4311403,"156,724,734.17","131,896,715.35","24,828,018.82",15
251,Eldorado do Sul,RS,4306767,"152,501,920.11","109,361,508.83","43,140,411.28",9


## 11. Gráfico de Barras Empilhadas: Danos Materiais vs. Prejuízos Públicos (Top 50)

**PERGUNTA SHOULD:**

Gráfico de barras empilhadas separando Danos Materiais (`DM_*`) e Prejuízos Públicos (`PEPL_*`) no Top 50

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# 1. Carregar o dataset
df = pd.read_csv('dataset_sul_hidrologico_2021_2025_filtrado.csv')

# ==============================================================================
# TRATAMENTO DE DADOS: Limpeza, Tratamento de NaN e Agrupamento
# ==============================================================================
col_mun = 'Nome_Municipio_Limpo' if 'Nome_Municipio_Limpo' in df.columns else 'Nome_Municipio'
col_uf = 'Sigla_UF'

# Tratamento da Chave Cadastral IBGE
if 'Cod_IBGE_Mun' in df.columns:
    df['Cod_IBGE_Mun'] = (
        df['Cod_IBGE_Mun']
        .fillna(0)
        .astype(int)
        .astype(str)
        .str.zfill(7)
    )

# Tratamento numérico (fillna para evitar valores 'em branco' na soma)
df['DM_limpo'] = pd.to_numeric(df['DM_total_danos_materiais'], errors='coerce').fillna(0)
df['PEPL_limpo'] = pd.to_numeric(df['PEPL_total_publico'], errors='coerce').fillna(0)
df['Custo_Dano_Evitado'] = df['DM_limpo'] + df['PEPL_limpo']

# Preenche nomes de município eventualmente nulos
df[col_mun] = df[col_mun].fillna('Não Identificado')

# Agrupamento das 50 maiores cidades por Custo_Dano_Evitado
ranking_top50 = (
    df.groupby([col_mun, col_uf], as_index=False)
    .agg(
        Custo_Dano_Evitado=('Custo_Dano_Evitado', 'sum'),
        DM_total_danos_materiais=('DM_limpo', 'sum'),
        PEPL_total_publico=('PEPL_limpo', 'sum'),
        Total_Eventos=('Custo_Dano_Evitado', 'count')
    )
    .sort_values(by='Custo_Dano_Evitado', ascending=False)
    .head(50)
)

# Para gráficos de barras horizontais, ordenar de forma crescente para o #1 ficar no topo
top50_plot = ranking_top50.sort_values('Custo_Dano_Evitado', ascending=True).copy()

# ==============================================================================
# FUNÇÃO DE FORMATAÇÃO DE MOEDA (Evita textos gigantes e poluídos na barra)
# ==============================================================================
def formata_moeda(v):
    if v <= 0:
        return ""  # Oculta rótulos zerados para não poluir
    elif v >= 1e9:
        return f"R$ {v/1e9:,.2f} bi"
    elif v >= 1e6:
        return f"R$ {v/1e6:,.1f} mi"
    else:
        return f"R$ {v:,.0f}"

# ==============================================================================
# CONSTRUÇÃO DO GRÁFICO PLOTLY
# ==============================================================================

# Rótulo amigável com Nome e UF ex: "São Leopoldo (RS)"
labels_y = top50_plot[col_mun] + ' (' + top50_plot[col_uf] + ')'

# Traço 1: Danos Materiais (Infraestrutura, Obras, Equipamentos)
trace1 = go.Bar(
    y=labels_y,
    x=top50_plot['DM_total_danos_materiais'],
    name='Danos Materiais',
    orientation='h',
    marker_color='#C0392B', # Vermelho elegante
    text=[formata_moeda(v) for v in top50_plot['DM_total_danos_materiais']],
    textposition='auto',
    insidetextanchor='middle',
    hovertemplate='<b>%{y}</b><br>Danos Materiais: R$ %{x:,.2f}<extra></extra>'
)

# Traço 2: Prejuízos Públicos (Serviços Essenciais, Socorro, Limpeza)
trace2 = go.Bar(
    y=labels_y,
    x=top50_plot['PEPL_total_publico'],
    name='Prejuízos Públicos',
    orientation='h',
    marker_color='#3498DB', # Azul elegante
    text=[formata_moeda(v) for v in top50_plot['PEPL_total_publico']],
    textposition='auto',
    insidetextanchor='middle',
    hovertemplate='<b>%{y}</b><br>Prejuízos Públicos: R$ %{x:,.2f}<extra></extra>'
)

data = [trace1, trace2]

layout = go.Layout(
    barmode='stack',  # Empilha as barras
    title=dict(
        text='<b>Top 50 Municípios: Composição de Danos Materiais e Prejuízos Públicos</b><br>'+
             '<span style="font-size:13px;color:#6B6B6B">Desastres hidrológicos, RS/SC/PR, 2021–2025</span>',
        x=0.02, xanchor='left'
    ),
    xaxis=dict(
        title='Valor Total (R$)',
        showgrid=True,
        gridcolor='#F0F0F0',
        zeroline=False,
        range=[0, top50_plot['Custo_Dano_Evitado'].max() * 1.15] # Margem para os rótulos
    ),
    yaxis=dict(
        title='',
        showgrid=False,
        automargin=True,
        dtick=1 # Garante a exibição de todas as 50 cidades
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=True,
    legend=dict(x=0.98, y=0.02, xanchor='right', yanchor='bottom', bgcolor='rgba(255,255,255,0.8)'),
    height=1400, # Altura para acomodar confortavelmente as 50 barras sem sobreposição
    margin=dict(l=180, r=40, t=80, b=60)
)

fig = go.Figure(data=data, layout=layout)
fig.show()

## 12. Contagem de Frequência de Ocorrências por Município

In [ ]:
# 1. Calcular a frequência de ocorrências por município
contagem_ocorrencias = df_recorte['Nome_Municipio_Limpo'].value_counts().reset_index()
contagem_ocorrencias.columns = ['Nome_Municipio_Limpo', 'Contagem']

# Adicionar Sigla_UF para melhor contexto (usando o df original para mapear)
# Primeiro, criar um DataFrame auxiliar para mapeamento, garantindo unicidade
map_municipio_uf = df_recorte[['Nome_Municipio_Limpo', 'Sigla_UF']].drop_duplicates()
contagem_ocorrencias = contagem_ocorrencias.merge(map_municipio_uf, on='Nome_Municipio_Limpo', how='left')

# Ordenar para o gráfico
contagem_ocorrencias = contagem_ocorrencias.sort_values('Contagem', ascending=True)

# Definir um formatador simples para a contagem
def formatador_contagem(v):
    return f"{int(v)} eventos"

# 2. Gerar o gráfico de barras usando a função 'grafico_barra_historia'
grafico_barra_historia(
    categorias=(contagem_ocorrencias['Nome_Municipio_Limpo'] + ' (' + contagem_ocorrencias['Sigla_UF'] + ')').tolist(),
    valores=contagem_ocorrencias['Contagem'].tolist(),
    titulo="Municípios com maior número de ocorrências de desastres hidrológicos",
    subtitulo="Frequência de registros por município (RS/SC/PR, 2021–2025)",
    formatador=formatador_contagem,
    height=1200
)